%md

# Control de parámetros de extracción

## Propósito

Obtener y validar el corte de extracción correspondiente al pipeline antes de iniciar el proceso de ingesta, generando además un identificador único para la ejecución.

## Flujo

1. Leer la tabla Delta de control.
2. Obtener la configuración del pipeline.
3. Validar la existencia de un único corte.
4. Validar la consistencia del rango de fechas.
5. Generar un `id_ejecucion` único para la ejecución.
6. Publicar `fecha_inicio`, `fecha_fin` e `id_ejecucion` mediante Task Values.

## Tabla de control

`workspace.metadata.pipeline_control`

## Parámetros publicados

* `fecha_inicio`
* `fecha_fin`
* `id_ejecucion`

Estos valores son consumidos por las siguientes Tasks del Workflow para mantener la trazabilidad de extremo a extremo.

## Dependencias

Este notebook debe ejecutarse como una **Task dentro de un Databricks Workflow** para que los Task Values puedan ser consumidos por las siguientes Tasks.

## Resultado esperado

Un único corte de extracción validado y un `id_ejecucion` generado para identificar la ejecución completa del pipeline.



In [0]:
# ============================================================
# INICIO DE LOG ETL
# ============================================================

import uuid
from pyspark.sql import functions as F

TABLA_ETL_LOG = "workspace.metadata.etl_log"

log_id = str(uuid.uuid4())

df_log_inicio = spark.createDataFrame(
    [
        (
            log_id,
            id_ejecucion,
            "00_prepare_pipeline_control",
            "metadata",
            None,
            None,
            0,
            "EN_PROCESO",
            None
        )
    ],
    """
    log_id STRING,
    id_ejecucion STRING,
    notebook STRING,
    capa STRING,
    fecha_inicio TIMESTAMP,
    fecha_fin TIMESTAMP,
    filas_procesadas BIGINT,
    estado STRING,
    mensaje_error STRING
    """
)

df_log_inicio = (
    df_log_inicio
    .withColumn("fecha_inicio", F.current_timestamp())
)

df_log_inicio.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(TABLA_ETL_LOG)

print(f"✓ ETL Log iniciado")
print(f"✓ log_id: {log_id}")
print(f"✓ id_ejecucion: {id_ejecucion}")
print("✓ Estado: EN_PROCESO")

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.metadata
""")

In [0]:
spark.sql("SHOW SCHEMAS IN workspace").show(truncate=False)

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.metadata.pipeline_control (
    pipeline_id INT,
    pipeline_name STRING,
    task_name STRING,
    load_type STRING,
    load_frequency STRING,
    last_successful_end_date DATE,
    next_start_date DATE,
    next_end_date DATE,
    status STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
""")

In [0]:
df_control = spark.read.table(
    "workspace.metadata.pipeline_control"
)

display(df_control)

In [0]:
from datetime import datetime

data = [
    (
        1,
        "databricks_extract_performance_logistico_dataco",
        "ingest_sharepoint_bronze",
        "incremental",
        "weekly",
        "2015-12-31",
        "2016-01-01",
        "2016-01-07",
        "pending",
        datetime.now(),
        datetime.now()
    )
]

columns = [
    "pipeline_id",
    "pipeline_name",
    "task_name",
    "load_type",
    "load_frequency",
    "last_successful_end_date",
    "next_start_date",
    "next_end_date",
    "status",
    "created_at",
    "updated_at"
]

df_initial = spark.createDataFrame(data, columns)

display(df_initial)

In [0]:
from pyspark.sql import functions as F

df_initial = (
    df_initial
    .withColumn(
        "last_successful_end_date",
        F.to_date("last_successful_end_date")
    )
    .withColumn(
        "next_start_date",
        F.to_date("next_start_date")
    )
    .withColumn(
        "next_end_date",
        F.to_date("next_end_date")
    )
)

df_initial.printSchema()

In [0]:
# Verificar si el pipeline ya existe
pipeline_exists = (
    spark.table("workspace.metadata.pipeline_control")
    .filter(F.col("pipeline_id") == 1)
    .limit(1)
    .count() > 0
)

if not pipeline_exists:
    (
        df_initial.write
        .format("delta")
        .mode("append")
        .saveAsTable("workspace.metadata.pipeline_control")
    )
    print("Pipeline inicializado correctamente.")
else:
    print("El pipeline ya existe. No se realizará ninguna inserción.")

In [0]:
df_control = spark.read.table(
    "workspace.metadata.pipeline_control"
)

display(df_control)

In [0]:
df_control = spark.read.table(
    "workspace.metadata.pipeline_control"
)

In [0]:
from pyspark.sql import functions as F

df_corte = (
    df_control
    .filter(
        F.col("pipeline_name") == "databricks_extract_performance_logistico_dataco"
    )
    .select(
        "next_start_date",
        "next_end_date"
    )
)

display(df_corte)

In [0]:
corte = df_corte.first()

fecha_inicio = corte["next_start_date"]
fecha_fin = corte["next_end_date"]

print(f"Fecha inicio de extracción: {fecha_inicio}")
print(f"Fecha fin de extracción: {fecha_fin}")

In [0]:
cantidad_cortes = df_corte.count()

if cantidad_cortes == 0:
    raise ValueError(
        "No existe un corte configurado para el pipeline."
    )

if cantidad_cortes > 1:
    raise ValueError(
        f"Se encontraron {cantidad_cortes} cortes para el pipeline. "
        "Debe existir exactamente uno."
    )

print("Validación OK: existe un único corte para el pipeline.")

In [0]:
if fecha_inicio is None:
    raise ValueError(
        "El campo next_start_date está vacío."
    )

if fecha_fin is None:
    raise ValueError(
        "El campo next_end_date está vacío."
    )

if fecha_inicio > fecha_fin:
    raise ValueError(
        f"El rango de fechas es inválido: "
        f"{fecha_inicio} > {fecha_fin}"
    )

print(
    f"Validación OK: rango válido "
    f"{fecha_inicio} → {fecha_fin}"
)

In [0]:
# ============================================================
# GENERACIÓN DE ID DE EJECUCIÓN
# ============================================================

import uuid

id_ejecucion = str(uuid.uuid4())

print(f"ID de ejecución: {id_ejecucion}")

In [0]:
dbutils.jobs.taskValues.set(
    key="fecha_inicio",
    value=str(fecha_inicio)
)

In [0]:
dbutils.jobs.taskValues.set(
    key="fecha_fin",
    value=str(fecha_fin)
)

In [0]:
dbutils.jobs.taskValues.set(

    key="id_ejecucion",

    value=id_ejecucion

)

In [0]:
print("Parámetros publicados correctamente.")
print(f"Fecha inicio: {fecha_inicio}")
print(f"Fecha fin:    {fecha_fin}")
print(f"ID ejecución: {id_ejecucion}")

In [0]:
# ============================================================
# REGISTRO DE EJECUCIÓN EN TABLA DE CONTROL
# ============================================================

from pyspark.sql import functions as F

TABLA_EJECUCION_CONTROL = "workspace.metadata.ejecucion_control"

# Crear la tabla si todavía no existe
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLA_EJECUCION_CONTROL} (
    id_ejecucion STRING,
    fecha_inicio DATE,
    fecha_fin DATE,
    fecha_inicio_ejecucion TIMESTAMP,
    fecha_fin_ejecucion TIMESTAMP,
    estado STRING,
    es_ejecucion_valida BOOLEAN
)
USING DELTA
""")

# Registrar la nueva ejecución como EN_PROCESO
df_ejecucion = spark.createDataFrame(
    [
        (
            id_ejecucion,
            fecha_inicio,
            fecha_fin,
            None,
            None,
            "EN_PROCESO",
            False
        )
    ],
    """
    id_ejecucion STRING,
    fecha_inicio DATE,
    fecha_fin DATE,
    fecha_inicio_ejecucion TIMESTAMP,
    fecha_fin_ejecucion TIMESTAMP,
    estado STRING,
    es_ejecucion_valida BOOLEAN
    """
)

df_ejecucion = (
    df_ejecucion
    .withColumn("fecha_inicio_ejecucion", F.current_timestamp())
)

df_ejecucion.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(TABLA_EJECUCION_CONTROL)

print(f"✓ Ejecución registrada: {id_ejecucion}")
print(f"✓ Corte: {fecha_inicio} → {fecha_fin}")
print("✓ Estado: EN_PROCESO")

In [0]:
# ============================================================
# VALIDACIÓN DE EJECUCIÓN REGISTRADA
# ============================================================

display(
    spark.table("workspace.metadata.ejecucion_control")
    .filter(F.col("id_ejecucion") == id_ejecucion)
)

In [0]:
%sql
SELECT *
FROM workspace.metadata.pipeline_control
ORDER BY created_at;

In [0]:
# ============================================================
# FIN DE LOG ETL - EXITOSO
# ============================================================

from delta.tables import DeltaTable

delta_log = DeltaTable.forName(
    spark,
    TABLA_ETL_LOG
)

delta_log.update(
    condition=f"log_id = '{log_id}'",
    set={
        "fecha_fin": "current_timestamp()",
        "estado": "'EXITOSO'",
        "filas_procesadas": "0",
        "mensaje_error": "NULL"
    }
)

print("✓ ETL Log finalizado correctamente")
print(f"✓ log_id: {log_id}")
print("✓ Estado: EXITOSO")